# SAIA2163 Final Project - Toxic Comment Detector

**Dataset:** Jigsaw Toxic Comment Classification Challenge  
**Task:** Binary classification - toxic vs non-toxic  
**Checkpoint 1:** Data loading, preprocessing pipeline, baseline model

## 1. Setup & Imports

In [1]:
import nltk, string, regex
import polars as pl
import numpy as np
import pandas as pd
from nltk.tokenize import word_tokenize, sent_tokenize, RegexpTokenizer
from nltk.corpus import stopwords as nltk_stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer
from collections import Counter
from langdetect import detect, LangDetectException
from tqdm import tqdm
from multiprocessing import Pool, cpu_count
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [2]:
nltk.download("stopwords")
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("omw-1.4", quiet=True)


[nltk_data] Downloading package stopwords to /home/mel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## 2. Data Loading & Exploration

In [3]:
Dataset = pl.read_csv("/home/mel/studies/Y2S2/NLP-SAIA2163/exercise/group-assignment/jigsaw-toxic-comment-classification-challenge/train.csv")
print(Dataset.shape)
print(Dataset.head(3))


(159571, 8)
shape: (3, 8)
┌───────────────┬───────────────┬───────┬──────────────┬─────────┬────────┬────────┬───────────────┐
│ id            ┆ comment_text  ┆ toxic ┆ severe_toxic ┆ obscene ┆ threat ┆ insult ┆ identity_hate │
│ ---           ┆ ---           ┆ ---   ┆ ---          ┆ ---     ┆ ---    ┆ ---    ┆ ---           │
│ str           ┆ str           ┆ i64   ┆ i64          ┆ i64     ┆ i64    ┆ i64    ┆ i64           │
╞═══════════════╪═══════════════╪═══════╪══════════════╪═════════╪════════╪════════╪═══════════════╡
│ 0000997932d77 ┆ Explanation   ┆ 0     ┆ 0            ┆ 0       ┆ 0      ┆ 0      ┆ 0             │
│ 7bf           ┆ Why the edits ┆       ┆              ┆         ┆        ┆        ┆               │
│               ┆ made…         ┆       ┆              ┆         ┆        ┆        ┆               │
│ 000103f0d9cfb ┆ D'aww! He     ┆ 0     ┆ 0            ┆ 0       ┆ 0      ┆ 0      ┆ 0             │
│ 60f           ┆ matches this  ┆       ┆              ┆         

In [4]:
info_test=pl.DataFrame({
    "Column": Dataset.columns,
    "Dtype": [str(dtype) for dtype in Dataset.dtypes],
    "Non-Null Count": [len(Dataset) - count for count in Dataset.null_count().to_numpy()[0]]
})
print(info_test)
Dataset["comment_text"].str.len_chars().describe()


shape: (8, 3)
┌───────────────┬────────┬────────────────┐
│ Column        ┆ Dtype  ┆ Non-Null Count │
│ ---           ┆ ---    ┆ ---            │
│ str           ┆ str    ┆ u32            │
╞═══════════════╪════════╪════════════════╡
│ id            ┆ String ┆ 159571         │
│ comment_text  ┆ String ┆ 159571         │
│ toxic         ┆ Int64  ┆ 159571         │
│ severe_toxic  ┆ Int64  ┆ 159571         │
│ obscene       ┆ Int64  ┆ 159571         │
│ threat        ┆ Int64  ┆ 159571         │
│ insult        ┆ Int64  ┆ 159571         │
│ identity_hate ┆ Int64  ┆ 159571         │
└───────────────┴────────┴────────────────┘


statistic,value
str,f64
"""count""",159571.0
"""null_count""",0.0
"""mean""",394.073221
"""std""",590.720282
"""min""",6.0
"""25%""",96.0
"""50%""",205.0
"""75%""",435.0
"""max""",5000.0


In [5]:
Dataset["toxic"].value_counts()

toxic,count
i64,u32
1,15294
0,144277


## 3. Preprocessing Pipeline

### 3.1 Remove Null & Empty Rows

In [6]:
# Extract as list for NLP
toxic_corpus = list(zip(
    Dataset["comment_text"].to_list(),
    Dataset["toxic"].to_list()
))

In [7]:
# Save the non-whitespace
Dataset = Dataset.filter(
    pl.col("comment_text").is_not_null() &
    (pl.col("comment_text").str.strip_chars() != "")
)

Dataset[["comment_text","toxic"]].write_csv("/home/mel/studies/Y2S2/NLP-SAIA2163/exercise/group-assignment/toxic_corpus.csv")
print(f"Saved toxic_corpus.csv: {Dataset.shape[0]} rows")


Saved toxic_corpus.csv: 159571 rows


In [8]:
print("After null filter:", Dataset.shape)

After null filter: (159571, 8)


### 3.2 Language Detection

*Retained for documentation. Language filtering skipped since Jigsaw is ~98% English. Non-English rows retained as noise impact is negligible.*

In [9]:
def safe_detect(text):
    try:
        if not text or text.strip() == "":
            return "unknown"
        return detect(text)
    except LangDetectException:
        return "unknown"
    except Exception:
        return "unknown"

"""
Dataset = Dataset.with_columns(
    pl.col("comment_text").map_elements(
        safe_detect,
        return_dtype=pl.String
    ).alias("language")
)
print(Dataset["language"].value_counts().sort("count", descending=True))
"""

'\nDataset = Dataset.with_columns(\n    pl.col("comment_text").map_elements(\n        safe_detect,\n        return_dtype=pl.String\n    ).alias("language")\n)\nprint(Dataset["language"].value_counts().sort("count", descending=True))\n'

In [10]:
"""
# Keep only English rows
Dataset = Dataset.filter(
    pl.col("language") == "en"
)

print(Dataset.shape)
print(Dataset["language"].value_counts())
"""

'\n# Keep only English rows\nDataset = Dataset.filter(\n    pl.col("language") == "en"\n)\n\nprint(Dataset.shape)\nprint(Dataset["language"].value_counts())\n'

### 3.3 Text Cleaning

Removes URLs, HTML/wiki markup, hex colors, CSS properties, keyboard spam, and special characters.

In [11]:
def clean_text(text):
    if not text or text.strip() == "":
        return ""

    # 1. Remove URLs
    text = regex.sub(r'https?://\S+', '', text)

    # 2. Remove HTML/Wiki markup
    text = regex.sub(r'\{\|.*?\|\}', '', text, flags=regex.DOTALL)  # wiki tables
    text = regex.sub(r'\|[\w\s=""""#;:]+', '', text)                # wiki style attrs
    text = regex.sub(r'style="""".*?""""', '', text)                 # style attributes
    text = regex.sub(r'<.*?>', '', text)                             # HTML tags

    # 3. Remove hex colors
    text = regex.sub(r'#[0-9a-fA-F]{3,6}', '', text)

    # 4. Remove CSS properties
    text = regex.sub(r'\b(background-color|vertical-align|padding|border|font-size|rowspan|colspan|height|width)\b.*?;', '', text)

    # 5. Remove keyboard spam — words longer than 25 chars with no vowel pattern
    words = text.split()
    words = [w for w in words if not (len(w) > 25 and regex.search(r'[^aeiou]{10,}', w.lower()))]
    text = " ".join(words)

    # 6. Remove pure numbers and number-heavy tokens
    text = regex.sub(r'\b\d+[\d.,]+\b', '', text)

    # 7. Remove leftover symbols
    text = regex.sub(r'[^\w\s]', ' ', text)

    # 8. Remove extra whitespace
    text = regex.sub(r'\s+', ' ', text).strip()

    return text


In [12]:
texts_raw = Dataset["comment_text"].to_list()

with Pool(cpu_count()) as pool:
    cleaned_results = list(tqdm(
        pool.imap(clean_text, texts_raw, chunksize=500),
        total=len(texts_raw),
        desc="Cleaning"
    ))

Dataset = Dataset.with_columns(
    pl.Series("cleaned_text", cleaned_results, dtype=pl.String)
)

# Remove empty after cleaning
Dataset = Dataset.filter(
    pl.col("cleaned_text").is_not_null() &
    (pl.col("cleaned_text").str.strip_chars() != "")
)
print(f"After cleaning: {Dataset.shape[0]} rows")


Cleaning: 100%|██████████| 159571/159571 [00:00<00:00, 164250.35it/s]


After cleaning: 159400 rows


### 3.4 Slang Detection

Identifies unusual tokens for manual review and slang map construction.

In [13]:
def slang_word(text):
    try:
        unusual = []
        words = regex.findall(r'\b\w+\b', text.lower())

        for word in words:
            if len(word) < 3:
                continue
            if (
                regex.search(r'(.)\1{1,}', word) or
                regex.search(r'[a-z]+[0-9]+[a-z]*', word) or
                regex.search(r'[0-9]+[a-z]+', word) or
                regex.search(r'(.)\1{2,}', word)
            ):
                unusual.append(word)

        return ", ".join(unusual) if unusual else ""
    except:
        return ""

In [14]:
texts = Dataset["cleaned_text"].to_list()

with Pool(cpu_count()) as pool:
    results = list(tqdm(
        pool.imap(slang_word, texts),
        total=len(texts)
    ))

Dataset = Dataset.with_columns(
    pl.Series("unusual_words", results)
)

review_df = Dataset.filter(
    pl.col("unusual_words") != ""
)[["comment_text", "unusual_words","toxic"]]

print(review_df.head(10))

100%|██████████| 159400/159400 [00:10<00:00, 14568.07it/s]


shape: (10, 3)
┌─────────────────────────────────┬─────────────────────────────────┬───────┐
│ comment_text                    ┆ unusual_words                   ┆ toxic │
│ ---                             ┆ ---                             ┆ ---   │
│ str                             ┆ str                             ┆ i64   │
╞═════════════════════════════════╪═════════════════════════════════╪═══════╡
│ Explanation                     ┆ metallica, dolls                ┆ 0     │
│ Why the edits made…             ┆                                 ┆       │
│ D'aww! He matches this backgro… ┆ aww, seemingly                  ┆ 0     │
│ Hey man, I'm really not trying… ┆ really, seems, formatting       ┆ 0     │
│ "                               ┆ suggestions, accidents, need, … ┆ 0     │
│ More                            ┆                                 ┆       │
│ I can't make any real s…        ┆                                 ┆       │
│ "                               ┆ well, tools, 

In [15]:
# Saving slang word in csv
review_df.write_csv("/home/mel/studies/Y2S2/NLP-SAIA2163/exercise/group-assignment/slang_review.csv")
print(f"Rows to review: {review_df.shape[0]}")


Rows to review: 138197


### 3.5 Slang Normalization

Maps obfuscated toxic words to canonical forms before tokenization.

In [16]:
slang_map = {
    # faggot variants
    'faggt': 'faggot',
    'faggo': 'faggot',
    'fagget': 'faggot',
    'faggit': 'faggot',
    'fagg': 'faggot',
    'faggets': 'faggot',
    'faggoty': 'faggot',
    'faggish': 'faggot',
    # fuck variants
    'fukk': 'fuck',
    'fukken': 'fuck',
    'fukking': 'fuck',
    'fukker': 'fuck',
    'fukkin': 'fuck',
    'fukked': 'fuck',
    'fuuck': 'fuck',
    'fuucker': 'fuck',
    'fuckk': 'fuck',
    'fuckoff': 'fuck',
    'fuckking': 'fuck',
    # shit variants
    'shiit': 'shit',
    'shitt': 'shit',
    'shitty': 'shit',
    'shittin': 'shit',
    'shitter': 'shit',
    # nigger variants
    'niggaz': 'nigger',
    'niggah': 'nigger',
    'niggars': 'nigger',
    'nigga': 'nigger',
    'niggas': 'nigger',
    'niggerlover': 'nigger',
    'niggering': 'nigger',
    'niggerism': 'nigger',
    'niggerfaggot': 'nigger',
    'sandnigger': 'nigger',
    'sandniggers': 'nigger',
    # compound ass words
    'dumbass': 'dumb',
    'jackass': 'jerk',
    'asshole': 'asshole',
    'assholes': 'asshole',
    'asshat': 'asshole',
    'asswipe': 'asshole',
    'assclown': 'asshole',
    'assbitch': 'asshole',
    'bitchass': 'asshole',
    'cuntass': 'asshole',
    'punkass': 'asshole',
    'lameass': 'asshole',
    'gayass': 'asshole',
    'fatass': 'asshole',
    'smartass': 'asshole',
    'badass': 'asshole',
    # pussy variants
    'pussyfuck': 'pussy',
    'pussylicker': 'pussy',
    'pussyass': 'pussy',
    # misc
    'goddamned': 'goddamn',
    'goddamnit': 'goddamn',
    'bullshitter': 'bullshit',
    'bullshitting': 'bullshit',
    'pissfuck': 'fuck',
    'buttfuck': 'fuck',
    'killyou': 'kill',
}

### 3.6 Tokenization, Stopword Removal, Stemming & Lemmatization

In [17]:
stop_words = set(nltk_stopwords.words("english"))
tokenizer = RegexpTokenizer(r"(?u)\w+")
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


In [18]:
import nltk

def get_wordnet_pos(tag):
    """Map POS tag to WordNet POS for lemmatization."""
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

def preprocessing(corpus):
    corpus = corpus.lower()
    tokens = tokenizer.tokenize(corpus)
    tokens = [slang_map.get(token, token) for token in tokens]
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [stemmer.stem(t) for t in tokens]
    pos_tags = nltk.pos_tag(tokens)
    tokens = [lemmatizer.lemmatize(t, get_wordnet_pos(pos)) for t, pos in pos_tags]
    return tokens


In [19]:
with Pool(cpu_count()) as pool:
    Preprocess_data = list(tqdm(
        pool.imap(preprocessing, texts),
        total=len(texts),
        desc="Preprocessing"
    ))
print(Preprocess_data[:5])

Preprocessing: 100%|██████████| 159400/159400 [00:22<00:00, 7139.67it/s] 


[['explan', 'edit', 'make', 'usernam', 'hardcor', 'metallica', 'fan', 'revert', 'vandal', 'closur', 'ga', 'vote', 'new', 'york', 'doll', 'fac', 'plea', 'remov', 'templat', 'talk', 'page', 'sinc', 'retir'], ['aww', 'match', 'background', 'colour', 'seemingli', 'stick', 'thank', 'talk', 'januari', 'utc'], ['hey', 'man', 'realli', 'tri', 'edit', 'war', 'guy', 'constantli', 'remov', 'relev', 'inform', 'talk', 'edit', 'instead', 'talk', 'page', 'seem', 'care', 'format', 'actual', 'info'], ['make', 'real', 'suggest', 'improv', 'wonder', 'section', 'statist', 'later', 'subsect', 'type', 'accid', 'think', 'refer', 'may', 'need', 'tidi', 'exact', 'format', 'ie', 'date', 'format', 'etc', 'later', 'one', 'el', 'first', 'prefer', 'format', 'style', 'refer', 'want', 'pleas', 'let', 'know', 'appear', 'backlog', 'articl', 'review', 'guess', 'may', 'delay', 'review', 'turn', 'list', 'relev', 'form', 'eg', 'wikipedia', 'good_article_nomin', 'transport'], ['sir', 'hero', 'chanc', 'rememb', 'page']]


In [20]:
# Join tokens back to string to save as CSV
Dataset = Dataset.with_columns(
    pl.Series("preprocessed", Preprocess_data)
)

Dataset = Dataset.with_columns(
    pl.col("preprocessed").map_elements(
        lambda x: " ".join(x),
        return_dtype=pl.String
    ).alias("preprocessed_text")
)

print(Dataset[["comment_text", "preprocessed_text","toxic"]].head(5))

shape: (5, 3)
┌─────────────────────────────────┬─────────────────────────────────┬───────┐
│ comment_text                    ┆ preprocessed_text               ┆ toxic │
│ ---                             ┆ ---                             ┆ ---   │
│ str                             ┆ str                             ┆ i64   │
╞═════════════════════════════════╪═════════════════════════════════╪═══════╡
│ Explanation                     ┆ explan edit make usernam hardc… ┆ 0     │
│ Why the edits made…             ┆                                 ┆       │
│ D'aww! He matches this backgro… ┆ aww match background colour se… ┆ 0     │
│ Hey man, I'm really not trying… ┆ hey man realli tri edit war gu… ┆ 0     │
│ "                               ┆ make real suggest improv wonde… ┆ 0     │
│ More                            ┆                                 ┆       │
│ I can't make any real s…        ┆                                 ┆       │
│ You, sir, are my hero. Any cha… ┆ sir hero chanc

## 4. Save Preprocessed Data

In [21]:
# Drop nulls before saving
Dataset = Dataset.filter(pl.col("preprocessed_text").is_not_null())

Dataset[["preprocessed_text", "toxic"]].write_csv("/home/mel/studies/Y2S2/NLP-SAIA2163/exercise/group-assignment/preprocessed_corpus.csv")
print(f"Saved preprocessed_corpus.csv: {Dataset.shape[0]} rows")
print(Dataset[["preprocessed_text", "toxic"]].head(3))


Saved preprocessed_corpus.csv: 159400 rows
shape: (3, 2)
┌─────────────────────────────────┬───────┐
│ preprocessed_text               ┆ toxic │
│ ---                             ┆ ---   │
│ str                             ┆ i64   │
╞═════════════════════════════════╪═══════╡
│ explan edit make usernam hardc… ┆ 0     │
│ aww match background colour se… ┆ 0     │
│ hey man realli tri edit war gu… ┆ 0     │
└─────────────────────────────────┴───────┘


## 5. Class Balancing

In [22]:
# Class balancing — undersample majority class to 50/50
import polars as pl

df_full = pl.read_csv("/home/mel/studies/Y2S2/NLP-SAIA2163/exercise/group-assignment/preprocessed_corpus.csv")
df_full = df_full.filter(pl.col("preprocessed_text").is_not_null())

toxic = df_full.filter(pl.col("toxic") == 1)
clean = df_full.filter(pl.col("toxic") == 0)

clean_sampled = clean.sample(n=toxic.shape[0], seed=42)
balanced = pl.concat([toxic, clean_sampled]).sample(fraction=1.0, shuffle=True, seed=42)

balanced.write_csv("/home/mel/studies/Y2S2/NLP-SAIA2163/exercise/group-assignment/balanced_corpus.csv")
print(f"Balanced dataset: {balanced.shape}")
print(balanced["toxic"].value_counts())

Balanced dataset: (30576, 2)
shape: (2, 2)
┌───────┬───────┐
│ toxic ┆ count │
│ ---   ┆ ---   │
│ i64   ┆ u32   │
╞═══════╪═══════╡
│ 1     ┆ 15288 │
│ 0     ┆ 15288 │
└───────┴───────┘


## 6. Baseline Model. Logistic Regression + TF-IDF

Trained on balanced dataset (15,288 toxic / 15,288 clean). Evaluated with accuracy, precision, recall, F1-score, and confusion matrix.

In [23]:
# Load balanced dataset
df = pd.read_csv("/home/mel/studies/Y2S2/NLP-SAIA2163/exercise/group-assignment/balanced_corpus.csv")
df = df.dropna(subset=['preprocessed_text'])

texts_final = df["preprocessed_text"].tolist()
labels = df["toxic"].tolist()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    texts_final, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

# Vectorize + Train
tfidf = TfidfVectorizer(max_features=10000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = model.predict(X_test_tfidf)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=["Non-Toxic", "Toxic"]))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8994
              precision    recall  f1-score   support

   Non-Toxic       0.87      0.93      0.90      3057
       Toxic       0.93      0.87      0.90      3057

    accuracy                           0.90      6114
   macro avg       0.90      0.90      0.90      6114
weighted avg       0.90      0.90      0.90      6114

[[2852  205]
 [ 410 2647]]


In [24]:
import pandas as pd
df = pd.read_csv("/home/mel/studies/Y2S2/NLP-SAIA2163/exercise/group-assignment/balanced_corpus.csv")
df = df.dropna(subset=['preprocessed_text'])
print(df.shape)
print(df['toxic'].value_counts())

(30568, 2)
toxic
1    15286
0    15282
Name: count, dtype: int64
